# Optimize spatiotemporal synthetic data parameters

This notebook searches for a spatiotemporal synthetic data-generating process such that:

- the **IID** engine declares equivalence,
- the **cluster-aware** engine declares equivalence,
- the **spatial** engine declares equivalence, and
- the **spatiotemporal** engine does **not** declare equivalence.

Unlike the interim validation-only version, this notebook uses the shipped
`optimize_spatiotemporal_params` optimizer module and saves a fresh
`best_spatiotemporal_params.json`.


## 1. Environment setup


In [ ]:
from pyTOST.data_gen.notebook_utils import (
    configure_notebook_environment,
    summarize_result,
    history_frame,
    result_payload,
)
from pyTOST.data_gen.optimize_spatiotemporal_params import (
    SearchSpace,
    search,
    save_best,
    validate_with_full_engine,
)

REPO_ROOT = configure_notebook_environment()
ALPHA = 0.05
MARGIN = 1.0
SEARCH_SEED = 123
VALIDATION_SEED = 2026
OUTPUT_JSON = "best_spatiotemporal_params.json"
BOOTSTRAP_B = 200

ENGINE_SPECS = {
    "IID": {"ci": "ci_iid", "eq": "eq_iid", "mu": "mu_hat_iid"},
    "Cluster": {"ci": "ci_cluster", "eq": "eq_cluster", "mu": "mu_hat_cluster"},
    "Spatial": {"ci": "ci_spatial", "eq": "eq_spatial", "mu": "mu_hat_spatial"},
    "Spatiotemporal": {"ci": "ci_spatiotemporal", "eq": "eq_spatiotemporal", "mu": "mu_hat_spatiotemporal"},
}


## 2. Configure the optimization problem


In [ ]:
space = SearchSpace()
N_ITER = 60
VERBOSE_EVERY = 10
NU = 2.5


## 3. Run the search


In [ ]:
best, history = search(
    seed=SEARCH_SEED,
    n_iter=N_ITER,
    alpha=ALPHA,
    margin=MARGIN,
    space=space,
    verbose_every=VERBOSE_EVERY,
    nu=NU,
)

best_summary = summarize_result(best, ENGINE_SPECS)
best_summary


## 4. Summarize the best candidate


In [ ]:
best_payload = result_payload(
    params=best.params,
    summary_source=best,
    engine_specs=ENGINE_SPECS,
    extra={"validation_seed": VALIDATION_SEED, "bootstrap_B": BOOTSTRAP_B, "nu": NU},
)
best_payload


## 5. Save the best parameters


In [ ]:
save_best(best, OUTPUT_JSON)
OUTPUT_JSON


## 6. Validate by regenerating data and rerunning the target engines


In [ ]:
validation = validate_with_full_engine(
    best.params,
    seed=best.params.get("_eval_seed", VALIDATION_SEED),
    alpha=ALPHA,
    margin=MARGIN,
    bootstrap_B=BOOTSTRAP_B,
)
validation_summary = summarize_result(validation, ENGINE_SPECS)
validation_summary


## 7. Search diagnostics


In [ ]:
history_df = history_frame(
    history,
    ENGINE_SPECS,
    param_keys=["n_space", "n_time", "n_clusters", "length_scale", "rho", "spatial_sd", "obs_sd", "delta"],
)
history_df.sort_values("score").head(10)
